# Part 3: Presentation Layer (`3_app_ui.ipynb`)
Interactive Gradio Web Application.


In [ ]:
import pandas as pd
import gradio as gr


In [ ]:
# ============================================================
# PART 3: PRESENTATION LAYER (INTERACTIVE APP & INFERENCE)
# ============================================================

data_df = pd.read_csv("maharashtra_analyzed.csv")

def get_business_recommendations(opp, emp_rate, bhk1):
    opp_str = str(opp).lower()
    recs = []
    if "high opportunity" in opp_str:
        recs.append("1. Retail & FMCG General Store: Low commercial rents and steady everyday footfall.")
        recs.append("2. Food Mess & Tiffin Service: Caters directly to growing local workforce.")
        recs.append("3. Cold Storage / Agro-Processing: High return due to strong agrarian connectivity.")
        if bhk1 < 20:
            recs.append("4. Early Land & Real Estate Investment: Maximum capital appreciation expected.")
    elif "developing" in opp_str:
        recs.append("1. Coaching & Skill Training Center: High demand from expanding youth population.")
        recs.append("2. E-Commerce Logistics Delivery Hub: Strategic distribution access across taluka.")
        recs.append("3. Modern Gym & Fitness Center: Rapidly rising disposable income among residents.")
        if emp_rate > 70:
            recs.append("4. Quick Service Restaurant (QSR) / Cafe: Strong evening youth gathering spots.")
    else:
        recs.append("1. Premium Diagnostics & Polyclinic: Dense population requires high-end healthcare.")
        recs.append("2. Boutique Cafe & Co-Working Hub: Established tech and white-collar professional base.")
        recs.append("3. Tech Consulting & IT Services: Skilled talent pool and high digital infrastructure.")
        recs.append("4. High-End Retail Franchise: High per-capita spending and lifestyle orientation.")
    return "\n".join(recs)

def search_location(query):
    if not query or not query.strip():
        return "Please enter a location name (e.g., Mumbai, Pune, Hinjawadi, Versova).", "", "", "", "", "", "", ""
    
    q = query.strip().lower()
    matches = data_df[
        data_df["Village"].str.lower().str.contains(q, na=False) |
        data_df["Taluka"].str.lower().str.contains(q, na=False) |
        data_df["District"].str.lower().str.contains(q, na=False)
    ]
    
    if matches.empty:
        return f"No records found matching '{query}'. Please search for a valid Maharashtra location.", "", "", "", "", "", "", ""
    
    matched_count = len(matches)
    avg_bhk1 = round(matches["1_BHK_Price"].mean(), 2)
    avg_bhk2 = round(matches["2_BHK_Price"].mean(), 2)
    avg_bhk3 = round(matches["3_BHK_Price"].mean(), 2)
    avg_emp = round(matches["Emp_Rate"].mean(), 2)
    avg_unemp = round(matches["Unemp_Rate"].mean(), 2)
    avg_fut = round(matches["Predicted_5Yr_Price"].mean(), 2)
    profit = round(avg_fut - avg_bhk1, 2)
    roi = round((profit / avg_bhk1) * 100, 1)
    
    # Most common opportunity cluster
    top_opp = matches["Business_Opportunity"].mode()[0]
    
    status_header = f"Found {matched_count} matching location(s) in Maharashtra for: '{query.title()}'"
    curr_str = f"Rs. {avg_bhk1} Lakhs"
    fut_str = f"Rs. {avg_fut} Lakhs"
    roi_str = f"+ Rs. {profit} Lakhs ({roi}% ROI)"
    emp_str = f"Employment: {avg_emp}% | Unemployment: {avg_unemp}%"
    other_str = f"2 BHK: Rs. {avg_bhk2} Lakhs | 3 BHK: Rs. {avg_bhk3} Lakhs"
    recommendations = get_business_recommendations(top_opp, avg_emp, avg_bhk1)
    
    return status_header, curr_str, fut_str, roi_str, emp_str, str(top_opp), other_str, recommendations

# Build Gradio UI
with gr.Blocks(title="Maharashtra Geo-Economic & Real Estate Analyzer") as demo:
    gr.Markdown("# Maharashtra Geo-Economic Analyzer & ROI Predictor")
    gr.Markdown("**AI/ML Practical Project** | Powered by K-Means Clustering & Linear Regression")
    
    with gr.Row():
        search_input = gr.Textbox(
            label="Search District, Taluka, or Village",
            placeholder="Type Mumbai, Pune, Hinjawadi, Nagpur, Versova, Nashik...",
            lines=1
        )
        search_btn = gr.Button("Analyze Location", variant="primary")
    
    status_text = gr.Markdown("### Ready to search. Type any Maharashtra location above.")
    
    with gr.Row():
        bhk1_box = gr.Textbox(label="Average 1 BHK Price", interactive=False)
        fut_box = gr.Textbox(label="ML Predicted 5-Year Price", interactive=False)
        roi_box = gr.Textbox(label="Expected Capital Gain & ROI", interactive=False)
    
    with gr.Row():
        emp_box = gr.Textbox(label="Employment / Unemployment Statistics", interactive=False)
        opp_box = gr.Textbox(label="K-Means Opportunity Tier", interactive=False)
        other_box = gr.Textbox(label="2 BHK & 3 BHK Benchmarks", interactive=False)
    
    biz_box = gr.Textbox(
        label="AI-Generated Business & Commercial Opportunities",
        interactive=False,
        lines=5
    )
    
    gr.Markdown("---")
    gr.Markdown("*Separation of Concerns: Data Ingestion (Layer 1) -> ML Training (Layer 2) -> Web Presentation (Layer 3)*")
    
    search_btn.click(
        fn=search_location,
        inputs=search_input,
        outputs=[status_text, bhk1_box, fut_box, roi_box, emp_box, opp_box, other_box, biz_box]
    )
    search_input.submit(
        fn=search_location,
        inputs=search_input,
        outputs=[status_text, bhk1_box, fut_box, roi_box, emp_box, opp_box, other_box, biz_box]
    )

print("Launching Gradio Web App...")
demo.launch(inbrowser=True)
